# A-WAN WP-3 — OPV2V real-data pipeline (Colab)
Real multi-view driving frames (OPV2V / OpenCOOD test_culvercity split, ~4 GB)
through the same VLM->facts->overlap pipeline. Produces `opv2v_results.json` —
drop into `runs/` as the real-data calibration artifact.
**You must first** download the split from the OpenCOOD data page (Google
Drive / UCLA Box links; accept their terms) into `opv2v/test_culvercity/`.

In [ ]:
!pip -q install "transformers==4.57.1" accelerate pillow num2words pyyaml
import pathlib
assert pathlib.Path("opv2v/test_culvercity").exists(), "download the split first (see cell above)"

In [ ]:
import yaml, json, numpy as np, torch, pathlib, re
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
MID="HuggingFaceTB/SmolVLM2-500M-Video-Instruct"
proc=AutoProcessor.from_pretrained(MID)
model=AutoModelForImageTextToText.from_pretrained(MID,dtype=torch.bfloat16).to("cuda").eval()
PROMPT=("Driving scene. List vehicles and pedestrians you can see. Reply ONLY JSON: "
        "{\"facts\":[{\"object\":\"car|truck|bus|person\",\"attr\":\"<color or type>\",\"confidence\":0.0}]}")
def perceive(img):
    msgs=[{"role":"user","content":[{"type":"image","image":img},{"type":"text","text":PROMPT}]}]
    inp=proc.apply_chat_template(msgs,add_generation_prompt=True,tokenize=True,return_dict=True,return_tensors="pt").to("cuda")
    with torch.no_grad(): out=model.generate(**inp,do_sample=False,max_new_tokens=160)
    return proc.decode(out[0][inp["input_ids"].shape[1]:],skip_special_tokens=True)

In [ ]:
root=pathlib.Path("opv2v/test_culvercity")
scen=sorted(root.iterdir())[0]
cavs=sorted(d for d in scen.iterdir() if d.is_dir())[:4]
frames=sorted(set(f.stem.split("_")[0] for f in cavs[0].glob("*.yaml")))[:6]
results=[]
for fr in frames:
    per_agent={}
    for cav in cavs:
        yml=yaml.safe_load(open(cav/(fr+".yaml")))
        img=Image.open(cav/(fr+"_camera0.png"))
        raw=perceive(img)
        m=re.search(r"\{.*\}",raw,re.S)
        facts=json.loads(m.group(0)).get("facts",[]) if m else []
        per_agent[cav.name]={"facts":facts,"n_gt":len(yml.get("vehicles",{})),"pose":yml["lidar_pose"]}
    results.append({"frame":fr,"agents":per_agent})
json.dump(results,open("opv2v_results.json","w"))
print("frames:",len(results))
from google.colab import files; files.download("opv2v_results.json")